# 02 – Data cleaning

**Goal:** turn the raw government tables into simple, analysis-ready CSV files in `data/processed/`.

Problems found in notebook 01 and fixed here:
- The vehicle tables mix several geography levels (UK, regions, counties, districts). Only **lower-tier local authorities** are kept, so no vehicle is counted twice.
- VEH0105 is in **thousands** and VEH0142 is in **numbers**. Both are converted to numbers.
- Quarters are stored as columns (*wide* format). They are reshaped to one row per area and quarter (*long* format).
- **Barnsley and Sheffield** use new 2024 codes in the charger data and old codes elsewhere. The old codes are replaced with the new ones.
- Special values such as `[x]` (not available) become empty values (`NaN`).
- **Company vehicles are registered at the company's address, not where the driver lives.** Lease and fleet companies are based in a few places, so totals are badly distorted there. For example, Windsor and Maidenhead has about 200,000 battery electric cars (more than its population), and 98% of them are company cars. Privately kept vehicles are therefore kept as separate columns for the local analysis.

**Important: two different charger measures.** EVCI0102 counts *chargers* (about 121,000 in the UK in July 2026). The historic table EVCI9001 counts *devices* (about 95,000), and one device can have several chargers. The two are never mixed in one calculation:
- the infrastructure gap (EVs per charger) uses EVCI0102, July 2026,
- growth over time uses EVCI9001, from October 2019.

DfT also notes that about 1,270 private chargers were removed from the data in June 2026, so the April to July 2026 change is not treated as real growth.

In [1]:
import pandas as pd

RAW = "../data/raw/"
PROCESSED = "../data/processed/"

# Code prefixes of lower-tier local authorities in England, Wales, Scotland and Northern Ireland
LOWER_TIER = ("E06", "E07", "E08", "E09", "W06", "S12", "N09")

# Barnsley and Sheffield received new codes in 2024 (old code -> new code)
CODE_FIX = {"E08000016": "E08000038", "E08000019": "E08000039"}

## 1. Licensed vehicles by local authority (VEH0105 + VEH0142)

The two vehicle tables have the same layout, so one small function cleans both. It keeps:
- lower-tier local authorities only,
- all keepers (`Keepership = Total`), and separately privately kept vehicles only (`Private`),
- two body types: **Cars** and **all vehicles** (`Total`),
- one fuel type at a time.

Then it reshapes the quarter columns into rows.

In [2]:
veh0105 = pd.read_excel(RAW + "veh0105.ods", sheet_name="VEH0105", header=4)
veh0142 = pd.read_excel(RAW + "veh0142.ods", sheet_name="VEH0142", header=4)

In [3]:
def vehicles_to_long(df, fuel, keepership, value_name):
    rows = df[
        df["ONS Code"].astype(str).str[:3].isin(LOWER_TIER)
        & (df["Keepership"] == keepership)
        & df["BodyType"].isin(["Cars", "Total"])
        & (df["Fuel"] == fuel)
    ]
    quarter_cols = [c for c in df.columns if " Q" in str(c)]
    long = rows.melt(
        id_vars=["ONS Code", "ONS Geography", "BodyType"],
        value_vars=quarter_cols,
        var_name="quarter",
        value_name=value_name,
    )
    # "[x]" (not available) becomes NaN
    long[value_name] = pd.to_numeric(long[value_name], errors="coerce")
    return long


all_vehicles = vehicles_to_long(veh0105, "Total", "Total", "all_vehicles")
all_vehicles_private = vehicles_to_long(veh0105, "Total", "Private", "all_vehicles_private")
# VEH0105 is in thousands: convert to number of vehicles
all_vehicles["all_vehicles"] = (all_vehicles["all_vehicles"] * 1000).round()
all_vehicles_private["all_vehicles_private"] = (all_vehicles_private["all_vehicles_private"] * 1000).round()

bev = vehicles_to_long(veh0142, "Battery electric", "Total", "bev")
bev_private = vehicles_to_long(veh0142, "Battery electric", "Private", "bev_private")
plugin = vehicles_to_long(veh0142, "Total", "Total", "plugin")

keys = ["ONS Code", "BodyType", "quarter"]
vehicles = all_vehicles
for extra, col in [(bev, "bev"), (plugin, "plugin"), (all_vehicles_private, "all_vehicles_private"), (bev_private, "bev_private")]:
    vehicles = vehicles.merge(extra[keys + [col]], on=keys, how="left")
vehicles.head()

,ONS Code,ONS Geography,BodyType,quarter,all_vehicles,bev,plugin,all_vehicles_private,bev_private
0,E06000047,County Durham,Cars,2026 Q2,257411.0,9547.0,13522.0,243774.0,7662.0
1,E06000005,Darlington,Cars,2026 Q2,60704.0,2026.0,3128.0,51710.0,1507.0
2,E06000001,Hartlepool,Cars,2026 Q2,43385.0,1362.0,2002.0,41031.0,1029.0
3,E06000002,Middlesbrough,Cars,2026 Q2,63780.0,1681.0,2582.0,60140.0,1207.0
4,E06000057,Northumberland,Cars,2026 Q2,175187.0,7170.0,10424.0,167398.0,5974.0


In [4]:
vehicles = vehicles.rename(columns={
    "ONS Code": "la_code",
    "ONS Geography": "la_name",
    "BodyType": "body_type",
})
vehicles["la_code"] = vehicles["la_code"].replace(CODE_FIX)
vehicles["la_name"] = vehicles["la_name"].str.strip()
vehicles["body_type"] = vehicles["body_type"].replace({"Total": "All vehicles"})

# Quarter end date, e.g. "2026 Q2" -> 2026-06-30, useful for charts
vehicles["quarter_end"] = pd.PeriodIndex(vehicles["quarter"].str.replace(" ", ""), freq="Q").end_time.date

vehicles = vehicles[["la_code", "la_name", "body_type", "quarter", "quarter_end",
                     "all_vehicles", "bev", "plugin", "all_vehicles_private", "bev_private"]]
print(vehicles.shape)
vehicles[vehicles["quarter"] == "2026 Q2"].head()

(48374, 10)


,la_code,la_name,body_type,quarter,quarter_end,all_vehicles,bev,plugin,all_vehicles_private,bev_private
0,E06000047,County Durham,Cars,2026 Q2,2026-06-30,257411.0,9547.0,13522.0,243774.0,7662.0
1,E06000005,Darlington,Cars,2026 Q2,2026-06-30,60704.0,2026.0,3128.0,51710.0,1507.0
2,E06000001,Hartlepool,Cars,2026 Q2,2026-06-30,43385.0,1362.0,2002.0,41031.0,1029.0
3,E06000002,Middlesbrough,Cars,2026 Q2,2026-06-30,63780.0,1681.0,2582.0,60140.0,1207.0
4,E06000057,Northumberland,Cars,2026 Q2,2026-06-30,175187.0,7170.0,10424.0,167398.0,5974.0


**Columns:** `all_vehicles` = all licensed vehicles, `bev` = battery electric, `plugin` = all plug-in vehicles (battery electric + plug-in hybrids). Columns ending in `_private` count privately kept vehicles only. All values are numbers of vehicles.

## 2. New car registrations, UK (VEH1153)

Cars only, whole UK, all keepers, annual and quarterly. The detailed fuel types are grouped into six simple groups.

In [5]:
veh1153 = pd.read_excel(RAW + "veh1153.ods", sheet_name="VEH1153a_RoadUsing", header=4)

uk_cars = veh1153[
    (veh1153["Geography"] == "United Kingdom")
    & (veh1153["BodyType"] == "Cars")
    & (veh1153["Keepership"] == "Total")
    & veh1153["Date Interval"].isin(["Annual", "Quarterly"])
]

new_registrations = pd.DataFrame({
    "period_type": uk_cars["Date Interval"].str.lower(),
    # "2014 Q3 (July to September)" -> "2014 Q3"
    "period": uk_cars["Date"].astype(str).str[:7],
    "petrol": uk_cars["Petrol"],
    "diesel": uk_cars["Diesel"],
    "hybrid": uk_cars["Hybrid Electric - Petrol"] + uk_cars["Hybrid Electric - Diesel"],
    "plugin_hybrid": uk_cars["Plug-in Hybrid Electric - Petrol"]
        + uk_cars["Plug-in Hybrid Electric - Diesel"]
        + uk_cars["Range Extended Electric"],
    "bev": uk_cars["Battery Electric"],
    "other": uk_cars["Fuel Cell Electric"] + uk_cars["Gas"] + uk_cars["Others"],
    "total": uk_cars["Total"],
})

# Values are in thousands: convert to number of cars
value_cols = ["petrol", "diesel", "hybrid", "plugin_hybrid", "bev", "other", "total"]
new_registrations[value_cols] = (new_registrations[value_cols] * 1000).round()
new_registrations = new_registrations.reset_index(drop=True)
new_registrations[new_registrations["period_type"] == "annual"]

,period_type,period,petrol,diesel,hybrid,plugin_hybrid,bev,other,total
0,annual,2015,1303010.0,1284010.0,44682.0,19163.0,10091.0,63.0,2661020.0
1,annual,2016,1339370.0,1293690.0,51697.0,28533.0,10429.0,51.0,2723770.0
2,annual,2017,1369350.0,1073540.0,72256.0,35224.0,13913.0,53.0,2564330.0
3,annual,2018,1489920.0,755625.0,88246.0,44362.0,15756.0,138.0,2394040.0
4,annual,2019,1540120.0,622089.0,110962.0,35277.0,38007.0,119.0,2346580.0
5,annual,2020,1007860.0,304443.0,168200.0,67112.0,107913.0,872.0,1656400.0
6,annual,2021,911684.0,194692.0,263804.0,114230.0,190420.0,2415.0,1677250.0
7,annual,2022,821258.0,127902.0,331439.0,100862.0,266875.0,3792.0,1652130.0
8,annual,2023,898808.0,126170.0,462801.0,141047.0,314222.0,2784.0,1945830.0
9,annual,2024,749208.0,110550.0,588954.0,168203.0,381869.0,1322.0,2000110.0


In [6]:
# Check: the fuel groups should add up to the total
difference = new_registrations[value_cols[:-1]].sum(axis=1) - new_registrations["total"]
print("Largest difference (cars):", difference.abs().max())

Largest difference (cars): 7.0


The largest difference is only a handful of cars, caused by rounding in the published thousands. The grouping is complete.

## 3. Public chargers by local authority, latest (EVCI0102)

Sheet `a` gives all public chargers and sheet `b` gives 50kW+ (rapid) chargers, for January, April and July 2026.

In [7]:
def chargers_to_long(sheet, value_name):
    df = pd.read_excel(RAW + "evci0102_2026-07_EV_chargers_by_local_authority_UK.ods", sheet_name=sheet, header=2)
    df.columns = ["la_code", "la_name", "2026-01-01", "2026-04-01", "2026-07-01"]
    return df.melt(id_vars=["la_code", "la_name"], var_name="snapshot_date", value_name=value_name)


chargers = chargers_to_long("EVCI0102a", "chargers").merge(
    chargers_to_long("EVCI0102b", "chargers_50kw_plus")[["la_code", "snapshot_date", "chargers_50kw_plus"]],
    on=["la_code", "snapshot_date"],
)
print(chargers.shape)
chargers.head()

(1083, 5)


,la_code,la_name,snapshot_date,chargers,chargers_50kw_plus
0,E07000223,Adur,2026-01-01,103,33
1,E07000032,Amber Valley,2026-01-01,102,30
2,E07000224,Arun,2026-01-01,273,46
3,E07000170,Ashfield,2026-01-01,100,32
4,E07000105,Ashford,2026-01-01,204,40


## 4. Public charging devices over time (EVCI9001, sheet 1a)

Quarterly devices per local authority since October 2019. The table also lists local authorities abolished in 2021 and 2023 (all `[x]` in recent quarters). These are dropped by keeping only the current local authorities from the charger table.

In [8]:
devices = pd.read_excel(RAW + "evci9001_2026-07_EV_charging_devices_UK.ods", sheet_name="1a", header=2)
devices = devices.rename(columns={
    "Local authority / region code": "la_code",
    "Local authority / region name": "la_name",
})
devices["la_code"] = devices["la_code"].replace(CODE_FIX)

# Keep the published UK total separately, for the national trend
devices_uk = devices[devices["la_code"] == "K02000001"].melt(
    id_vars=["la_code", "la_name"], var_name="quarter_start", value_name="devices"
)[["quarter_start", "devices"]]
devices_uk["quarter_start"] = pd.to_datetime(devices_uk["quarter_start"].str[:6], format="%b-%y").dt.date

current_las = set(chargers["la_code"])
devices = devices[devices["la_code"].isin(current_las)]

devices = devices.melt(id_vars=["la_code", "la_name"], var_name="quarter_start", value_name="devices")
# "Oct-19" -> 2019-10-01. The first 6 characters drop note markers such as "Oct-24 [Note 14]"
devices["quarter_start"] = pd.to_datetime(devices["quarter_start"].str[:6], format="%b-%y").dt.date
devices["devices"] = pd.to_numeric(devices["devices"], errors="coerce")
print(devices.shape, "| local authorities:", devices["la_code"].nunique())
print("Missing values:", devices["devices"].isna().sum())
devices.head()

(10108, 4) | local authorities: 361
Missing values: 74


,la_code,la_name,quarter_start,devices
0,E06000047,County Durham,2019-10-01,92.0
1,E06000005,Darlington,2019-10-01,27.0
2,E06000001,Hartlepool,2019-10-01,5.0
3,E06000002,Middlesbrough,2019-10-01,25.0
4,E06000057,Northumberland,2019-10-01,139.0


**Missing values:** the 74 empty values belong to six local authorities created in 2021 and 2023 (North and West Northamptonshire, Cumberland, Westmorland and Furness, North Yorkshire, Somerset). They have no figures for the quarters before they existed. Because of this, the **UK trend over time uses the published UK total** (saved separately as `devices_uk`), not the sum of local authorities.

## 5. Deprivation (IMD 2025, England only)

Two measures are kept:
- `imd_score`: average deprivation score (higher = more deprived),
- `imd_rank`: rank of that score among the 296 English local authorities (1 = most deprived).

In [9]:
imd = pd.read_excel(RAW + "File_10_IoD2025_Local_Authority_District_Summaries_lower-tier_v2.xlsx", sheet_name="IMD")
# Some column names end with a space, so remove it first
imd.columns = imd.columns.str.strip()

deprivation = imd.rename(columns={
    "Local Authority District code (2024)": "la_code",
    "Local Authority District name (2024)": "la_name",
    "IMD - Average score": "imd_score",
    "IMD - Rank of average score": "imd_rank",
})[["la_code", "la_name", "imd_score", "imd_rank"]]
deprivation["la_code"] = deprivation["la_code"].replace(CODE_FIX)
print(deprivation.shape)
deprivation.sort_values("imd_rank").head()

(296, 4)


,la_code,la_name,imd_score,imd_rank
8,E06000009,Blackpool,43.467,1
1,E06000002,Middlesbrough,40.037,2
136,E07000117,Burnley,38.681,3
229,E08000003,Manchester,38.677,4
250,E08000025,Birmingham,38.115,5


## 6. Income and population (ONS GDHI 2024)

From the same ONS file:
- Table 3: gross disposable household income (GDHI) per head, in pounds,
- Table 2: resident population, used later for chargers per 100,000 people.

The ONS region (or nation) of each local authority is also kept, as it is useful for grouping.

In [10]:
GDHI_FILE = RAW + "ons_gdhi_local_authorities_1997to2024.xlsx"
income = pd.read_excel(GDHI_FILE, sheet_name="Table 3", header=3)
population = pd.read_excel(GDHI_FILE, sheet_name="Table 2", header=3)

# Year columns are stored as text, e.g. "2024"
income = income[["Region", "LAD code", "LAD name", "2024"]].rename(columns={"2024": "gdhi_per_head_2024"})
population = population[["LAD code", "2024"]].rename(columns={"2024": "population_2024"})

income_population = income.merge(population, on="LAD code").rename(columns={
    "Region": "region",
    "LAD code": "la_code",
    "LAD name": "la_name",
})
income_population["la_code"] = income_population["la_code"].replace(CODE_FIX)
income_population = income_population[["la_code", "la_name", "region", "gdhi_per_head_2024", "population_2024"]]
print(income_population.shape)
income_population.head()

(361, 5)


,la_code,la_name,region,gdhi_per_head_2024,population_2024
0,E06000001,Hartlepool,North East,20080,98180
1,E06000004,Stockton-on-Tees,North East,20623,206800
2,E06000002,Middlesbrough,North East,17855,156161
3,E06000003,Redcar and Cleveland,North East,19739,139228
4,E06000005,Darlington,North East,21587,112489


## 7. Housing type (Census 2021, England and Wales)

Census table TS044 counts households by accommodation type. It is used later (question 4) as a rough sign of **home charging**: people in flats and terraced houses usually have no driveway, so they depend more on public chargers.

Two fixes are needed:
- The census uses 2021 boundaries. In 2023, 17 districts in Cumbria, North Yorkshire and Somerset were merged into 4 new unitary authorities. Household counts can simply be added up, so the old districts are summed into the new codes.
- Barnsley and Sheffield get their new codes (`CODE_FIX`).

Scotland and Northern Ireland run separate censuses with different tables, so they are not included.

In [11]:
census = pd.read_csv(RAW + "census2021-ts044-ltla.csv")
census.columns = census.columns.str.replace("Accommodation type: ", "")

# Old districts (2021) -> new unitary authority (2023)
MERGED_2023 = {
    "E07000026": "E06000063", "E07000028": "E06000063", "E07000029": "E06000063",  # Cumberland
    "E07000027": "E06000064", "E07000030": "E06000064", "E07000031": "E06000064",  # Westmorland and Furness
    "E07000163": "E06000065", "E07000164": "E06000065", "E07000165": "E06000065", "E07000166": "E06000065",
    "E07000167": "E06000065", "E07000168": "E06000065", "E07000169": "E06000065",  # North Yorkshire
    "E07000187": "E06000066", "E07000188": "E06000066", "E07000189": "E06000066", "E07000246": "E06000066",  # Somerset
}

housing = pd.DataFrame({
    "la_code": census["geography code"].replace(CODE_FIX).replace(MERGED_2023),
    "households": census["Total: All households"],
    "detached": census["Detached"],
    "semi_detached": census["Semi-detached"],
    "terraced": census["Terraced"],
    # All four kinds of flat or shared building
    "flats": census[[
        "In a purpose-built block of flats or tenement",
        "Part of a converted or shared house, including bedsits",
        "Part of another converted building, for example, former school, church or warehouse",
        "In a commercial building, for example, in an office building, hotel or over a shop",
    ]].sum(axis=1),
    "caravan_or_temporary": census["A caravan or other mobile or temporary structure"],
})
housing = housing.groupby("la_code", as_index=False).sum()

print("Rows before / after merging districts:", len(census), "/", len(housing))
print("Households before / after:", census["Total: All households"].sum(), "/", housing["households"].sum())
housing.head()

Rows before / after merging districts: 331 / 318
Households before / after: 24783191 / 24783191


,la_code,households,detached,semi_detached,terraced,flats,caravan_or_temporary
0,E06000001,40932,8086,14537,13469,4725,115
1,E06000002,60262,10389,25485,16768,7569,51
2,E06000003,61637,12449,27942,15666,5496,84
3,E06000004,83756,22209,35494,18298,7676,79
4,E06000005,48920,9331,19148,13937,6426,78


## 8. Checks before saving

1. Do all tables contain the same local authorities?
2. Are there duplicate rows or missing values in the latest period?
3. Do the local authority totals add up to the published UK totals?

In [12]:
# 1. Same local authorities everywhere (the charger table is the reference list)
england_las = {c for c in current_las if c.startswith("E")}
print("Local authorities in charger table:", len(current_las))
print("Vehicles: same list?        ", set(vehicles["la_code"]) == current_las)
print("Devices: same list?         ", set(devices["la_code"]) == current_las)
print("Income: same list?          ", set(income_population["la_code"]) == current_las)
print("Deprivation = England list? ", set(deprivation["la_code"]) == england_las)
print("Housing = England and Wales list?", set(housing["la_code"]) == {c for c in current_las if c[0] in "EW"})

Local authorities in charger table: 361
Vehicles: same list?         True
Devices: same list?          True
Income: same list?           True
Deprivation = England list?  True
Housing = England and Wales list? True


In [13]:
# 2. Duplicates and missing values
print("Duplicate rows:")
print("  vehicles   ", vehicles.duplicated(["la_code", "body_type", "quarter"]).sum())
print("  chargers   ", chargers.duplicated(["la_code", "snapshot_date"]).sum())
print("  devices    ", devices.duplicated(["la_code", "quarter_start"]).sum())

latest_vehicles = vehicles[vehicles["quarter"] == "2026 Q2"]
print("\nMissing values, vehicles 2026 Q2:", latest_vehicles[["all_vehicles", "bev", "plugin", "all_vehicles_private", "bev_private"]].isna().sum().to_dict())
print("Missing values, chargers:", chargers[["chargers", "chargers_50kw_plus"]].isna().sum().to_dict())
print("Missing values, income/population:", income_population.isna().sum().sum())
print("Missing values, deprivation:", deprivation.isna().sum().sum())
print("Missing values, housing:", housing.isna().sum().sum())

# Housing types should add up to all households
type_sum = housing[["detached", "semi_detached", "terraced", "flats", "caravan_or_temporary"]].sum(axis=1)
print("Housing types add up to total?", (type_sum == housing["households"]).all())

Duplicate rows:
  vehicles    0
  chargers    0
  devices     0

Missing values, vehicles 2026 Q2: {'all_vehicles': 0, 'bev': 0, 'plugin': 0, 'all_vehicles_private': 0, 'bev_private': 0}
Missing values, chargers: {'chargers': 0, 'chargers_50kw_plus': 0}
Missing values, income/population: 0
Missing values, deprivation: 0
Missing values, housing: 0
Housing types add up to total? True


In [14]:
# 3. Local authority totals vs published UK totals (battery electric cars, 2026 Q2)
uk_row = veh0142[
    (veh0142["ONS Code"] == "K02000001")
    & (veh0142["BodyType"] == "Cars")
    & (veh0142["Fuel"] == "Battery electric")
    & (veh0142["Keepership"] == "Total")
]
uk_bev_cars = uk_row["2026 Q2"].iloc[0]
la_bev_cars = latest_vehicles.loc[latest_vehicles["body_type"] == "Cars", "bev"].sum()

print(f"UK total (published):        {uk_bev_cars:,.0f}")
print(f"Sum of local authorities:    {la_bev_cars:,.0f}")
print(f"Not assigned to an area:     {uk_bev_cars - la_bev_cars:,.0f} ({(uk_bev_cars - la_bev_cars) / uk_bev_cars:.1%})")

UK total (published):        2,016,968
Sum of local authorities:    1,977,019
Not assigned to an area:     39,949 (2.0%)


The 2% gap is vehicles that DfT could not link to a local authority (the `[z]` rows seen in notebook 01). Most of them (about 31,000) are "vehicles under disposal", i.e. between keepers, for example sitting with a dealer. The rest have an unknown local authority. They are left out of the local analysis, which is a small and acceptable loss.

## 9. Save the clean tables

In [15]:
vehicles.to_csv(PROCESSED + "vehicles_by_la.csv", index=False)
new_registrations.to_csv(PROCESSED + "new_car_registrations_uk.csv", index=False)
chargers.to_csv(PROCESSED + "chargers_by_la.csv", index=False)
devices.to_csv(PROCESSED + "charging_devices_by_la_history.csv", index=False)
devices_uk.to_csv(PROCESSED + "charging_devices_uk_history.csv", index=False)
deprivation.to_csv(PROCESSED + "deprivation_by_la.csv", index=False)
income_population.to_csv(PROCESSED + "income_population_by_la.csv", index=False)
housing.to_csv(PROCESSED + "housing_by_la.csv", index=False)

import os
for f in sorted(os.listdir(PROCESSED)):
    if f.endswith(".csv"):
        print(f"{f:40s} {os.path.getsize(PROCESSED + f) / 1024:8.0f} KB")

chargers_by_la.csv                             43 KB
charging_devices_by_la_history.csv            387 KB
charging_devices_uk_history.csv                 0 KB
deprivation_by_la.csv                           9 KB
housing_by_la.csv                              13 KB
income_population_by_la.csv                    17 KB
la_analysis_table.csv                          46 KB
new_car_registrations_uk.csv                    4 KB
vehicles_by_la.csv                           3944 KB


## Summary

| File | One row per | Main columns |
| --- | --- | --- |
| `vehicles_by_la.csv` | local authority × body type × quarter | `all_vehicles`, `bev`, `plugin`, `all_vehicles_private`, `bev_private` |
| `new_car_registrations_uk.csv` | year or quarter (UK, cars) | `petrol`, `diesel`, `hybrid`, `plugin_hybrid`, `bev`, `other`, `total` |
| `chargers_by_la.csv` | local authority × snapshot (Jan/Apr/Jul 2026) | `chargers`, `chargers_50kw_plus` |
| `charging_devices_by_la_history.csv` | local authority × quarter (from Oct 2019) | `devices` |
| `charging_devices_uk_history.csv` | quarter (UK total, from Oct 2019) | `devices` |
| `deprivation_by_la.csv` | local authority (England) | `imd_score`, `imd_rank` |
| `income_population_by_la.csv` | local authority | `region`, `gdhi_per_head_2024`, `population_2024` |
| `housing_by_la.csv` | local authority (England and Wales, Census 2021) | `households`, `detached`, `semi_detached`, `terraced`, `flats` |

All tables share the same 361 local authority codes (296 in England), so they can be joined on `la_code`.

**Next step (notebook 03):** answer business question 1. How fast is the UK fleet shifting to battery electric vehicles?